# Hedging the guarantee: results in reading order

This notebook reads what the scripts wrote. It recomputes nothing, so it runs in a couple of
seconds and it cannot disagree with `reports/tables/`. Run `make all` first, or the steps named
at the head of each section.

The order is the order the argument runs in: what the contract is worth, how much of that is
assumption, what its risk looks like, whether the model matches Jackson's disclosure, how far the
proxy the hedge runs on can be trusted, what hedging does, what capital it leaves, and why
reported earnings still move when the economic hedge is working.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TABLES = ROOT / "reports" / "tables"
FIGURES = ROOT / "reports" / "figures"


def table(name):
    return pd.read_csv(TABLES / f"{name}.csv")


def figure(name):
    display(Image(filename=str(FIGURES / f"{name}.png")))


sorted(p.stem for p in TABLES.glob("*.csv"))

## 1. The contract at issue

`make valuation`. A $100,000 single premium at issue age 70, Perspective II with the Flex GMWB
Core rider, valued at 31 December 2025 under Heston volatility and a one-factor Hull-White short
rate.

The market risk benefit comes out at zero, which is a check rather than a coincidence: Note 6
says the benefit starts at a fair value of zero where attributed fees cover projected claims, and
the attribution percentage calibration produces is 0.786, so they do. The gross guarantee is
22.2% of premium and the death benefit above the account value is another 1.2%, which belongs in
the same valuation because it sits on the same account and moves the other way with longevity.

In [ ]:
table("valuation_headline")

Why the guarantee is worth a fifth of premium on a risk-neutral basis: the owner defers five
years, accruing the Core option's 6% bonus on the benefit base, and then draws 5.95% a year
against a risk-neutral drift of about 4.2% less 2.26% of charges. The account depletes on most
paths. First claims arrive in policy year 6, the flow peaks at year 17, and by year 20 the
account is gone on 78% of paths. The guarantee is the expected outcome for this contract rather
than a tail event.

In [ ]:
figure("cash_flow_profile")

In [ ]:
flows = table("cash_flow_profile")
flows.loc[flows["policy_year"] <= 25,
          ["policy_year", "survival", "expected_claim", "expected_death_claim", "expected_fee",
           "prob_exhausted", "mean_account_value", "mean_benefit_base"]].round(1)

## 2. How much of this is assumption

Two kinds of variation, and reading them the same way is a category error.

A **market or assumption** change happens after inception, when the attribution percentage is
already fixed, so what it moves is the benefit. A **design** change is a different contract: at
issue it would have been given its own percentage and its benefit would also start at zero, so
the thing that distinguishes it is the percentage its own fees have to attribute. A design whose
implied attribution is near one is a contract whose fees barely cover its guarantee.

In [ ]:
robustness = table("valuation_robustness")
display(robustness[robustness["comparison"] == "fixed attribution"]
        [["variation", "market_risk_benefit", "mrb_change", "mrb_pct_of_premium", "note"]].round(0))
display(robustness[robustness["comparison"] == "own attribution"]
        [["variation", "implied_attribution", "attribution_change", "gross_pct_of_premium",
          "note"]].round(4))

The rate shock moves the benefit most and the long-run volatility level comes next; the current
variance barely registers, because mean reversion at 4.8 takes it out within a year of a
forty-five-year liability. The truncation rows are on common draws, which is the only way they
mean anything: cutting the projection at age 115 costs nothing measurable and at 105 costs $36.

On the design side, a younger issue age and a high-expense fund are what make the rider dear - at
issue age 65 the Core option needs 0.96 of its attributable fees, against 0.79 at 70 - and
deferring improves the margin rather than worsening it, because ten years of rider charge on a
bonus-inflated benefit base outweighs the larger guarantee it buys.

In [ ]:
table("monte_carlo_convergence").round(2)

## 3. Risk

`make greeks`. Equity exposure is negative and large: 14.5% of account value has to be sold short
to neutralise it. Rho is negative because a long-dated liability discounts away faster when rates
rise. Each Greek carries the standard error of its own paired difference rather than of the two
levels, which is the only version of the number that can be checked.

In [ ]:
display(table("greeks").round(2))
display(table("hedge_sizing").round(2))

Almost all of the volatility exposure sits in the long-run level - 866 per point against 27 for
the current variance - and that is the parameter the calibration has least to go on. The chain
stops at 3.2 years and the projection runs 45, so the level doing the work is an extrapolation.
It is not a free one: pinning it to the realised ten-year level plus the issuer's one-point
margin, which is the methodology Note 6 describes, costs four times the volatility error beyond
two years and drives mean reversion to its bound. The robustness table prices that choice at
-2,328, or -2.3% of premium.

In [ ]:
figure("volatility_calibration")

Two Cboe index histories test that one-day surface across a decade without entering the
calibration. The term structure first: the chain reaches 3.23 years and the hedge's puts run one
and two, so the option leg's volatility is extended from a quoted tenor by the model's own
variance curve, and six months is as far as free data goes. The outward rows are the only test of
the direction that extension actually uses.

In [ ]:
table("volatility_curve_check").round(3)

Both outward rows come out low, and the reason is structural. Mean reversion at 4.80 is a
seven-week half-life, so the model has almost fully reverted by six months: it carries an 18.3%
one-month quote to 20.2% at six months where the market prices 21.7%. The puts inherit the same
mapping out to one and two years, so they are priced too cheaply and the 21.67% ten-year option
cost is a lower bound. Pointing the other way, it means the fitted long-run level of 21.68% looks
too *low* rather than too high - which is independent support, from data the calibration never
touched, for letting the chain reject the issuer's 19.12%.

The skew is the one thing no variance quote can see: every volatility index is a variance rate,
and rho and xi can trade off against each other without moving the expected average variance at
any tenor. Cboe's SKEW index reports `100 - 10 x` the risk-neutral skewness of the thirty-day
return, and the calibrated parameters generate that skewness in closed form off the characteristic
function.

In [ ]:
table("skew_check").round(4).T

The model's thirty-day left tail is about two thirds as deep as the market's and shallower on 80%
of the decade's days, which is what a pure diffusion does - reaching -3.5 over a month without
jumps needs a correlation near minus one and the chain does not ask for one. The direction cuts
against this project's own headline rather than for it: too little short-horizon left tail
understates the chance of the sharp declines that put a living benefit in the money, so it biases
the liability down, and the model already comes out around three times the disclosure.

The last two fields are mirror images by construction. The model's skewness falls out of one state
variable, so the model index is a monotone function of the one-month quote and its rank
correlation with the observed index is forced to be minus the quote's own. That is not weak
agreement, it is no information: whatever moves the market's skew day to day is not in this
model.

Value and sensitivity across moneyness. At issue the benefit base equals the account value; a
book that has ridden markets up sits to the left of that, which is where the disclosure sits.

In [ ]:
figure("moneyness_curve")

### Where equity exposure changes sign

Not every state leaves the insurer short the index. The benefit base steps up to the contract
value on the anniversary, so with the contract value above the benefit base the index level on
that one date fixes the guaranteed income for life, and both the claims leg and the fees leg
respond positively to the index.

Read off the regression proxy, because a fractional policy time only means something to its
pre-event and post-event fits, and restricted to policy years 5 and 9 where the proxy's delta can
be trusted. At 1.20 of the benefit base in policy year 5 the exposure runs from -27,492 with most
of the year to go to +2,679 on its eve, and changes sign with no market move behind it. A book
with anniversaries spread across the calendar averages this away; one policy does not, which is
one reason a single-policy turnover statistic overstates what a programme sees.

In [ ]:
table("step_up_proximity").round(1)

## 4. Against the disclosure

`make validate`. Jackson runs a 10% equity shock and a parallel rate shift every year in Item 7A.
Everything below is scaled by variable annuity account value, which is the only way one $100,000
policy and a $236bn book go on the same axis.

Sign first.

In [ ]:
table("sign_agreement")

Then shape. Where a filing discloses both 50bp and 100bp, the ratio of the two is a convexity test
the model was never fitted to. The filing gives 1.855 up and 2.128 down; the model gives 1.882 and
2.128 at a benefit base equal to the account value, and the ratio barely moves across moneyness.

In [ ]:
display(table("convexity_disclosed").round(4))
display(table("convexity_model").round(4))

Then scale. One policy rolled from its 2016 inception to each disclosed date, with the attribution
percentage fixed at inception. It calibrates to 1.00 there: a guarantee written at a 1.57%
ten-year rate did not have the fees to cover itself, which is the mechanism that later turns it
into a net asset.

Each date is valued twice, at the anniversary on or before it and at the following one, because
the recursion steps in whole policy years while the disclosed dates are year-ends. The spread
between the two is the size of that approximation and it is reported rather than assumed away.

In [ ]:
in_force = table("in_force_vs_disclosed")
primary = in_force[in_force["duration_basis"] == "anniversary on or before"]
primary[["as_of", "attained_age", "account_value", "gwb_over_av", "ten_year_zero_pct",
         "model_value_pct_av", "disclosed_value_pct_av",
         "model_equity_down_10pct", "disclosed_equity_down_10pct",
         "model_rates_up_100bp", "disclosed_rates_up_100bp"]].round(3)

The levels sit about three times the disclosure, which is what hypothesis 4 predicted, and the
*decline* matches almost exactly: the model falls 4.56 to 2.64 across the four years and the
disclosure falls 1.47 to 0.85, which is the same proportion to within a tenth of a per cent.

The vintage portfolio does the same comparison on five issue years valued as one book, each
carrying the attribution percentage its own issue date calibrates to. Four of the five calibrate
to 1.00; only the 2024 vintage, written at a 4.33% ten-year rate, has margin.

In [ ]:
detail = table("portfolio_vintage_detail")
display(detail[detail["as_of"] == detail["as_of"].max()]
        [["issue_date", "issue_age", "attained_age", "deferring", "gawa_pct", "account_value",
          "gwb_over_av", "attribution"]].round(3))
portfolio = table("portfolio_vs_disclosed")
display(portfolio[["as_of", "weighted_attained_age", "weighted_gawa_pct", "deferring_share",
                   "gwb_over_av", "net_amount_at_risk_pct_av", "model_value_pct_av",
                   "disclosed_value_pct_av"]].round(3))
display(table("portfolio_comparison_summary").round(3))

The multiple runs 2.28 to 3.48 across six shocks and four dates, and a multiple that near-constant
points at one structural assumption rather than a pile of errors. The candidate is the static
full-utilisation, no-lapse base case, which is the benchmark in the literature precisely because
it is the most expensive one for the insurer.

Sweeping it does not close the gap, and that is the result rather than a disappointment.

In [ ]:
behaviour = table("behaviour_reconciliation")
behaviour[["utilisation", "lapse_rate", "gwb_over_av", "ratio_equity_down_10pct",
           "ratio_equity_up_10pct", "ratio_rates_up_100bp", "ratio_rates_down_100bp",
           "equity_ratio", "rate_ratio", "equity_over_rate", "mean_ratio"]].round(3)

Across the grid the equity multiple falls 2.10 to 1.59 while the rate multiple falls 2.09 to 0.47.
Drawing less takes duration out of the guarantee - fewer paths exhaust, so it is less of a
long-dated annuity - and the rate sensitivity collapses with it. The benefit base is still there
whatever the owner draws, so the equity sensitivity barely moves. By the time utilisation is low
enough to match the rate figure the model is still 2.03 times the disclosed equity figure.

What is left for the equity half is moneyness, and the shock locator says so independently: the
disclosed figure falls on the model's own curve at a benefit base over account value of 0.94
declining to 0.82, with the up and down shocks implying the same ratio without being made to. The
rate shocks fall outside the model's range at every moneyness, which is the same statement in
reverse.

In [ ]:
located = table("disclosure_located_on_curve")
display(located.round(3))
figure("shock_comparison")

In [ ]:
table("withdrawal_rate_diagnostic").round(4)

## 5. How far the proxy can be trusted

`make proxy`. The hedge cannot run a full valuation at every rebalance date, so it runs a
least-squares regression proxy fitted on the simulated paths. Everything in section 6 inherits
that proxy's errors, so they are measured before the hedging results rather than after.

The proxy's **value** is accurate: an out-of-sample R-squared above 0.995 inside its design range
from policy year 1, and a root mean square error of two to seven tenths of a per cent of premium
against nested simulation. Tenths, not thousandths - the committed column is a share of premium
rather than a percentage of it, which this cell had the wrong way round.

Its **delta** is not accurate early. At policy year 1 the root mean square error is 12.6 against a
mean absolute nested delta of 35, which is 36%, and 21% in year 2. It is 10% by year 5 and 5% by
year 14, and then deteriorates again to 23% by year 25 - not because the fit gets worse, the value
error barely moves, but because the delta itself shrinks as the book runs off and the same
absolute error becomes a larger share of it. The backtest covers policy years 3 to 13, which is
the best part of that range, and nothing in this project reads a delta off policy year 1.

Its **second derivative** is not usable at all, which is why the option leg is sized from a
tabulated nested surface instead.

In [ ]:
display(table("proxy_accuracy")
        [["year", "r_squared_in_range", "rmse_share_of_premium_in_range",
          "delta_rmse_share_of_premium_in_range", "mean_abs_nested_delta",
          "share_outside_design_range"]].round(4))
display(table("hedge_convexity_source").round(4))

## 6. Hedging

`make hedge`. One policy, three years in force at the start, rolled along realised history from
September 2016 to September 2026 and rebalanced daily. The strategies add one instrument class at
a time: S1 futures, S2 adds a receive-fixed swap, S3 adds listed puts, with S4 to S6 varying the
option design, the hedge ratio and a macro overlay.

The crisis replays are chosen by what happened rather than by what flatters a hedge, and they are
deliberately different in kind.

In [ ]:
replays = table("hedge_crisis_replays")
replays[["episode", "strategy", "index_return", "rate_move_bp", "peak_implied_vol",
         "total_pnl_pct", "pnl_sd_pct", "worst_day_pct", "variance_reduction",
         "cost_pct"]].round(4)

February and March 2020 makes the point better than the summary statistics. The unhedged guarantee
lost 22.0% of account value over the episode. Futures alone recovered two thirds of that, futures
and swaps left 2.7%, and adding the option leg turned it into a 1.9% gain. For this liability that
episode was as much a convexity event as a level event.

The frontier is residual volatility against what it cost, swept over the cost assumption at half,
one and two times its base level, because a hedging result quoted at one cost assumption is a
result about that assumption.

In [ ]:
display(table("hedge_frequency_frontier").round(4))
figure("hedge_frontier")
figure("leg_comparison")

The option leg is where the cost is, and the put design sweep says the long-dated end of it is
not earning its keep: a quarter-year put at 0.90 of spot costs 7.7% of account value over the
decade against 21.2% for a one-year put at the same strike, for almost the same residual. The
comparison is run on three axes - the whole window, the covid window, and cost - because a design
that only wins on one of them has not won.

In [ ]:
design = table("hedge_put_design")
design[["tenor_years", "strike_over_spot", "pnl_sd_pct", "total_cost_pct", "total_pnl_pct",
        "stress_pnl_sd_pct", "stress_total_pnl_pct", "stress_worst_day_pct",
        "gamma_per_cost", "on_frontier"]].round(4)

Model risk is priced the same way: the world is whatever history did, and the hedge is sized from
Greeks that are deliberately wrong in a named way. The extra residual is the cost of calibration
error in hedging, which no amount of fit diagnostics can give.

In [ ]:
table("hedge_model_risk").round(4)

### What one decade is worth as evidence

`make real-world`. Every number above comes from one path. The bootstrap resamples the decade's
equity days into other orderings - keeping each day paired with its own volatility state, and
leaving the rate path on history's own course, because a reordered level is not a rate scenario -
and runs the same strategies over them in two drift arms.

In [ ]:
summary = table("real_world")
summary[summary["sample"] == "all"][
    ["arm", "strategy", "paths", "variance_removed_p10", "variance_removed_median",
     "variance_removed_p90", "realised_variance_removed", "realised_percentile",
     "residual_sd_median_pct", "cost_median_pct", "worst_day_improved_share"]].round(4)

The realised decade sits at the 93rd to 100th percentile on every strategy and both arms. The
delta-and-rho hedge removes 91% of the daily variance on the path that happened and a median of
81% across reorderings of the same days, so the backtest did not measure the hedge so much as the
hedge on a favourable ordering.

Two things survive and one does not. The ranking survives: the richer hedge is tighter on 90 to
100% of reorderings. The size of the put leg's edge does not - a median 0.041 points of residual
against 0.058 on the realised path, for twelve points of cost. And the worst day is improved on
only 63 to 73% of reorderings, which means that on 27 to 37% of them the hedged book's
single worst day is worse than the unhedged book's.


In [ ]:
display(table("real_world_ranking").round(5))
paths_run = table("real_world_paths")
paths_run.groupby("arm")[["outside_design_share", "index_end", "pnl_sd_pct"]].describe().round(3)

The extrapolation share is why the re-centred arm is the one to read. A bull market walks the
contract out of the money and out of the region the proxy was fitted in: the as-drawn arm asks the
proxy for a state outside its design on a median 43% of rebalances against 26% re-centred and 36%
on the realised path. The arm that is more defensible economically is also the one the model can
value.

### The one external check on any of this

`make offset`. Everything above revalues the liability with the model that sized the hedge, so a
high variance reduction is close to arithmetic. Item 7A is the only outside evidence there is, and
it is better than it looks: the same filings that publish the guarantee's response to a shock
publish the derivative book's response to the same shock on the same date, so the ratio is a
disclosed offset ratio and nothing here is fitted to it.

The sign convention is the part that bites. A liability impact is a change in a carrying amount
and a derivative impact is a change in an asset's mark, so the two cancel in earnings when their
raw figures agree - a complete hedge is +1, not -1.

In [ ]:
disclosed = table("disclosed_hedge_offset")
disclosed[["as_of", "liability_basis", "shock", "guarantee_impact_musd", "rila_impact_musd",
           "derivative_impact_musd", "offset_of_guarantee", "offset_of_both_lines",
           "rila_absorbs_of_guarantee", "both_lines_share_of_guarantee"]].round(3)

Four year-ends on the market risk benefit, which is one measurement throughout. The rate share
goes 37-43% at the end of 2022, to 78-86% a year later, and sits at 79-88% through 2024 and 2025.
The equity share is at or above a full hedge on the downside through 2023 and four fifths of one
on the upside - 104% and 77%, then 110% and 81% - and
halves to 49% and 50% in 2024, the first full year after Brooke Re was formed in December 2023.
The two legs moved in opposite directions and the dates say when.

The 2025 equity pair is the one that would mislead on its own. Taken against the guarantee alone
the offset falls to 9% and turns negative, which reads as a programme being dismantled. The same
filing discloses a fixed-index and RILA embedded derivative whose equity sensitivity went from
\$4m to \$1,321m over that year against a guarantee sensitivity of \$1,574m - a liability that
owes more when equity rises is a natural short against a guarantee that gets cheaper - and it
absorbs 79 to 84% of the equity move before any derivative is involved. On both lines the offset
is 49% then 44%. The hedge did not shrink; the net exposure did.

The caveat on that combined column is in the last field: once the two lines cancel it divides by
a fifth of the guarantee's down move and a sixth of its up move, and `-53%` is arithmetic rather
than a measurement.


In [ ]:
comparison = table("hedge_offset_comparison")
display(comparison[["as_of", "risk", "shock", "offset_of_guarantee", "model_offset_S1",
                    "model_offset_S2", "model_offset_S3"]].round(3))
figure("disclosed_offset")

An offset below one is not by itself a partial hedge: the liability is convex in rates and a swap
is nearly linear, so even a book sized to kill rho exactly under-recovers a large shift. That is
what the model's columns are for. A full hedge of the model's own liability at these dates lands
between 93 and 109% on rates and 90 and 113% on equity, so convexity buys about eight points of
scatter either side of one - not the twelve to twenty point rate shortfall or the fifty to a
hundred point equity one the disclosure shows outside
2022 and 2023.

2024-12-31 settles that argument rather than leaving it to the benchmark. It is disclosed at 50bp
by the FY2024 filing and at 100bp by the FY2025 one, and the disclosed ratio moves 1.1 points on
the down shift and 2.7 on the up where a full hedge of the model's liability moves 3.8 and 4.3.
The disclosed book's ratio is *less* sensitive to the size of the shift than a complete hedge is,
which is the opposite of what a convexity explanation would predict.

The futures-only column is the sanity check: an equity hedge should barely touch a rate shock, and
2 to 3% is the small rho the futures forward carries through its own discount factor.

All of which rests on a hand-transcribed table, so it is checked rather than trusted. Thirty-two
disclosed blocks, instrument lines reproducing every Total exactly; three balance-sheet dates
disclosed by two filings each with every overlapping figure agreeing; and both transcriptions
verified against the filings by hash rather than by rereading.

### E5. What the index-linked book absorbs before any hedge

`make netting`. Jackson owes the upside on a registered index-linked annuity and owns it on a
withdrawal guarantee, so part of the guarantee never needs an external hedge. The offset comparison
above measured that from the outside and ran into a figure nothing explained: the fixed-index and
RILA embedded derivative's equity sensitivity went from \$4m to \$1,321m over 2025, a factor of
330, on a balance that doubled.

The mechanism reads backwards from the product's own vocabulary. Differentiating the credited
return gives a slope of one between zero and the cap, zero above it, zero inside the buffer and one
again below it - a buffer absorbs the *first* slice of a loss and the holder bears everything past
it, so a segment through its buffer is still fully exposed. Only a capped segment has stopped
moving.

In [ ]:
figure("rila_netting")
surface = table("rila_exposure_surface")
surface.pivot_table(index="elapsed_years", columns="index_since_issue",
                    values="equity_exposure_pct_of_account").round(3)

Highest for a segment late in its term sitting just above where it was written, lowest once the
index has run through the cap. The whole surface spans a factor of ten, and a *book* - six-year
segments, a 20% buffer, caps swept, cohorts rolled along realised index history and weighted by the
issuance the filings imply - spans a factor of 2.4 across everything it can reach.

The level is what licenses the exposure. Nothing here was fitted to the filed embedded derivative,
and it comes out at the same magnitude moving the same way.

In [ ]:
display(table("rila_level_check").round(3))
table("rila_netting")[["as_of", "shock", "observed_rila_over_va",
                       "model_absorbed_at_observed_size", "disclosed_absorbed_share",
                       "filed_over_model"]].round(4)

A book of Jackson's own filed size should have absorbed 3.5%, 10.5%, 27% and 50% of the
guarantee's equity move across the four year-ends. The filings report 0.1%, 0.2%, 0.5% and 79% -
under two to three per cent of the modelled figure for three years, then 1.6 times it.

Nothing an index-linked book can do moves a sensitivity by 330 in a year when its whole reachable
range is a factor of 2.4. What would produce exactly this is a line carried net of the derivatives
hedging it through the FY2024 filing and gross from FY2025. The filings do not say that, and it is
not relied on for anything except the conclusion that follows either way: **the 2024 and 2025
combined-basis offset figures are not the same measurement**, so the flatness of that series across
those two years is an artefact and the market-risk-benefit column is the one to read.

## 7. What the hedge leaves in capital

`make statutory`. The economic hedge is sized on a risk-neutral liability. The statutory
requirement is a real-world tail measure with a surrender-value floor under it, and the two do not
move together.

In [ ]:
display(table("statutory_requirement").round(3))
display(table("statutory_floor_reach").round(3))
display(table("hedge_surrender_floor").round(4))

The requirement is reported against the equity risk premium rather than at one value, because
nothing in free data identifies it: CTE(70) runs 13.6% of premium at a 2% premium down to 7.5% at
6%, and CTE(90) runs 27.9% to 22.6%. The hedge removes most of the economic variance and much less
of the statutory variance, because the floor binds in the states a delta hedge is busy in.

## 8. Economic against reported

`make reporting replica`. The same hedge positions, marked on three bases: the economic liability
the hedge was sized on, the reporting liability with the annuity table's margins, and the
reporting liability discounted at Treasury plus the insurer's own non-performance spread.

In [ ]:
lens = table("reporting_lens")
lens[["strategy", "economic_total_pct", "net_income_total_pct", "oci_total_pct",
      "economic_sd_pct", "net_income_sd_pct", "net_income_sd_multiple", "oci_sd_pct",
      "comprehensive_sd_pct"]].round(4)

In [ ]:
figure("economic_versus_reported")

Reported net income is 9 to 15% more variable than the economic outcome under identical positions,
on every hedged strategy. The more interesting line is the own-credit adjustment: its daily
standard deviation is 0.127% of account value against 0.153% for the whole of S3's hedged net
income, and no hedge in the programme targets it, because under ASU 2018-12 its movement goes to
other comprehensive income rather than through net income.

Credit spreads widen when equity markets fall, so that piece is a natural offset to the guarantee
- and the rule books it outside net income. An accounting boundary turns an offset into a
mismatch, which is what Item 7A means when it says the company does not hedge its US GAAP
liabilities and that this has produced net income volatility.

Finally, the model's own hedge-against-liability offset against the one Jackson's filed XBRL
series shows.

In [ ]:
display(table("disclosure_replica").round(4))
display(table("disclosure_own_credit").round(4))

The filed quarterly series shows a correlation of +0.006 between the liability movement and the
hedge result over eighteen quarters, which at that sample size could not have detected anything
smaller than about 0.47. The model's own offset is near-tautological by construction and is here
as a counterfactual rather than as a result: it says what the filed series would look like if the
hedge targeted the reported liability, and it does not look like that.

## 8. What the rider earned, and the fee it needed

`make economics`. Everything above is about variance. None of it says whether the product made
money, and the rider charge is levied on the benefit base, so that is the basis this section is
quoted on throughout.

The split is an identity rather than an attribution. Over any window

    net = rider charge + interest on cash - trading and carry - the guarantee, net of the hedge

where the last term is claims paid plus the change in the guarantee's value less what the hedge
recovered. There is no residual for an error to hide in: the terms are a rearrangement of the cash
recursion and the ledger's own net worth, and across all seventy-five runs they reconcile to the
ledger's profit to within 2e-14 on a benefit base of 153.


In [ ]:
economics = table("rider_economics")
whole = economics[economics["scenario"] == "the whole replay"]
figure("rider_economics")
whole.pivot_table(index=["years_since_issue", "cohort"], columns="strategy",
                  values="net_bp").round(0)


Unhedged, the rider earned 174 to 529 basis points a year on the decade that happened. Every
hedged arm gave it back, which is what hedging a short-equity position through a 15.4%-a-year
market does rather than a verdict on hedging - the bootstrap already said the realised decade sat
at the 93rd to 100th percentile of its own reorderings.

The term worth looking at separately is the funding, because it is the one a fee-and-cost summary
leaves out.


In [ ]:
whole[whole["strategy"].isin(("S0", "S2"))][
    ["cohort", "strategy", "fee_income_bp", "cash_interest_bp", "hedge_cost_bp",
     "uncovered_cost_bp", "net_bp"]
].round(1)


The frontier puts the delta-and-rho hedge's cost over the decade at 0.08% of account value, and
that is right for what it measures: commission, spread and carry. Funding the losses that hedge
realised against a rising market ran 35 to 52 basis points of benefit base a year - a sixth to a
third of the entire rider charge, and a hundred to three hundred times the trading cost. The line
changes sign with the strategy: the unhedged book accumulates fees and earns interest, the hedged
books borrow and pay. In a falling decade it is a credit. Either way it is the large one.

Under stress the ranking holds and the level does not. A hedge carrying no convexity under-covers
a convex guarantee in a fall, so the crisis columns are negative even where the hedge itself
gained.


In [ ]:
economics[economics["strategy"] == "S2"].pivot_table(
    index=["years_since_issue", "cohort"], columns="scenario", values="net_bp"
).round(0)


An annual rate on an eleven-day window is a rate and not an outcome. The one positive cell
belongs to the cohort whose account had already fallen far enough that the guarantee had little
convexity left to miss.

### The loop closes the other way

The break-even fee was meant to close this. Off the September 2016 curve the solve refuses, and
the fee curve says why.


In [ ]:
curve = table("rider_fee_curve")
curve["charge_bp"] = (10000 * curve["fee_pct"]).round(0)
curve["guarantee_pct_of_premium"] = (100 * curve["value_pct_of_account"]).round(2)
display(curve[["charge_bp", "guarantee_pct_of_premium", "pv_claims", "pv_fees"]].round(2))
table("rider_break_even")[["issue_age", "income_start_age", "charged_pct", "fair_pct",
                           "margin_bp", "reason"]]


The guarantee gets cheaper as the charge rises, because the charge is revenue; it stops getting
cheaper near 5%, because by then the charge drains the account fast enough to bring the claim
forward and end the fee stream before the payments end; and it never reaches zero. Six of the eight
contracts across issue ages 65 to 75 admit no charge that prices them. The two that do are both
sold at 75 and both need more than they are paid.

The same engine on the December 2025 curve prices the same contract at a market risk benefit of
zero, with an attribution percentage of 0.786. The difference between the two is 264 basis points
of ten-year rate, 1.57% against 4.21%, and nothing else - same contract, same mortality, same
paths. On a guarantee whose payments run forty years past the valuation date that is the whole of
the economics, and it is the plainest statement here of why the rate half of Jackson's programme is
the half that doubled over 2023.

A last caution on reading the two halves together. The replay says the rider earned money unhedged
on the path that happened; the valuation says it was underpriced at the moment it was sold. Those
do not contradict each other - one is a realised outcome on a single favourable ordering of a
single favourable decade, the other a risk-neutral expectation across all of them - and neither is
a statement about what Jackson charged or earned, since the contract here is a stylised
reconstruction and the fee a published rate sheet figure.
